# Session 3: numerical inverse kinematics and a pick-and-place path
**A. Real-world problem.** A pick-and-place robot must follow a path: lift the part from
one spot, slide it to another. The path is a list of gripper positions; the motors need joint
angles for every one of them. For a 3-joint arm in the plane there is no single formula (one
spare joint), so we solve by iteration, like Newton's method. We also find out what goes
wrong at a fully stretched arm and when a target cannot be reached.

**Plan.** (1) the Jacobian, checked against finite differences, (2) damped least squares
iteration, (3) singularities and manipulability, (4) a smooth straight-line path,
(5) null-space joint-limit avoidance, (6) failure cases and sensitivity.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
LINKS = np.array([0.30, 0.25, 0.15])      # metres

## Step 1: forward kinematics and the Jacobian
x = sum L_k cos(c_k), y = sum L_k sin(c_k), with c_k = q_1 + ... + q_k. Joint i moves every
link from i onwards, so a small turn dq_i moves the gripper by
    d(x, y) = ( -sum_{k>=i} L_k sin(c_k),  sum_{k>=i} L_k cos(c_k) ) dq_i.
Collect these columns in the 2 x 3 Jacobian J, so that  d(position) = J dq.

In [ ]:
def fk(q, links=LINKS):
    c = np.cumsum(np.asarray(q, dtype=float))
    return np.array([np.sum(links * np.cos(c)), np.sum(links * np.sin(c))])


def jacobian(q, links=LINKS):
    c = np.cumsum(np.asarray(q, dtype=float))
    lx = np.asarray(links) * -np.sin(c)
    ly = np.asarray(links) * np.cos(c)
    # reverse cumulative sums: link k counts for every joint i <= k
    return np.array([np.cumsum(lx[::-1])[::-1], np.cumsum(ly[::-1])[::-1]])


def jacobian_fd(q, links=LINKS, h=1e-6):
    """The same matrix by finite differences (central), used only to check jacobian()."""
    q = np.asarray(q, dtype=float)
    cols = []
    for i in range(len(q)):
        dq = np.zeros(len(q))
        dq[i] = h
        cols.append((fk(q + dq, links) - fk(q - dq, links)) / (2 * h))
    return np.array(cols).T

## Step 2: damped least squares (Newton-style) inverse kinematics
If the gripper is off target by e = target - fk(q), we want dq with J dq = e. With 3 joints
and 2 equations there are many dq; the one with the smallest size is the pseudoinverse
J^T (J J^T)^-1 e. Near a singularity J J^T is almost not invertible, so we add lambda^2 I:
    dq = J^T (J J^T + lambda^2 I)^-1 e
Repeat until e is tiny. A step cap keeps the iteration from leaping far.

In [ ]:
def dls_step(J, e, lam):
    return J.T @ np.linalg.solve(J @ J.T + lam ** 2 * np.eye(J.shape[0]), e)


def solve_ik(target, q0, links=LINKS, lam=0.01, limit=None, tol=1e-9, max_iter=300, max_step=0.1):
    """Iterate q <- q + dq. Returns (q, final error in metres, iterations used).
    limit: joint limit in radians (every |q_i| <= limit, applied by clipping) or None."""
    q = np.array(q0, dtype=float)
    target = np.asarray(target, dtype=float)
    for it in range(max_iter):
        e = target - fk(q, links)
        if np.linalg.norm(e) < tol:
            return q, float(np.linalg.norm(e)), it
        dq = dls_step(jacobian(q, links), e, lam)
        big = np.max(np.abs(dq))
        if big > max_step:
            dq *= max_step / big
        q = q + dq
        if limit is not None:
            q = np.clip(q, -limit, limit)
    return q, float(np.linalg.norm(target - fk(q, links))), max_iter

## Step 3: singularities and manipulability
The singular values s1 >= s2 of J say how easily the gripper moves in its two most and least
movable directions. Their product w = s1 s2 = sqrt(det(J J^T)) is the manipulability. When
the arm is fully stretched, J loses rank: s2 = 0, w = 0 and the gripper cannot move along
the arm. A pure pseudoinverse then demands enormous joint speeds for any such motion.

In [ ]:
def singular_values(q, links=LINKS):
    return np.linalg.svd(jacobian(q, links), compute_uv=False)


def manipulability(q, links=LINKS):
    s = singular_values(q, links)
    return float(s[0] * s[1])


def joint_speeds(q, v, links=LINKS, lam=None):
    """Joint speeds (rad/s) for a wanted gripper velocity v (m/s): pseudoinverse if lam is None, else damped."""
    J = jacobian(q, links)
    if lam is None:
        return np.linalg.pinv(J, rcond=1e-15) @ np.asarray(v, dtype=float)
    return dls_step(J, np.asarray(v, dtype=float), lam)

## Step 4: a smooth straight-line path
Move from A to B on a straight line, but not at constant speed (the robot would jerk at the
start and stop). A quintic timing s(t) = 10 t^3 - 15 t^4 + 6 t^5, t in [0, 1], starts and
ends with zero speed AND zero acceleration. Each point of the path is turned into joint
angles by solve_ik, starting from the previous answer (a "warm start").

In [ ]:
def quintic(t):
    t = np.asarray(t, dtype=float)
    return 10 * t ** 3 - 15 * t ** 4 + 6 * t ** 5


def cubic(t):
    t = np.asarray(t, dtype=float)
    return 3 * t ** 2 - 2 * t ** 3


def line_path(a, b, n=41, timing=quintic):
    s = timing(np.linspace(0.0, 1.0, n))[:, None]
    return np.asarray(a, dtype=float) + s * (np.asarray(b, dtype=float) - np.asarray(a, dtype=float))


def track_path(points, q0, links=LINKS, lam=0.01, limit=None, null_gain=0.0, q_pref=None, null_iters=20, tol=1e-10):
    """Joint angles for every path point (warm start from the previous point).
    null_gain > 0 adds null-space motion towards q_pref (default: all zeros): the arm changes shape
    but the gripper does not move, because N = I - J+ J removes everything that would move it."""
    q = np.array(q0, dtype=float)
    pref = np.zeros(len(q)) if q_pref is None else np.asarray(q_pref, dtype=float)
    out = []
    for p in points:
        p = np.asarray(p, dtype=float)
        for k in range(null_iters if null_gain > 0 else 1):
            if null_gain > 0:
                J = jacobian(q, links)
                Jp = J.T @ np.linalg.inv(J @ J.T + lam ** 2 * np.eye(2))
                q = q + (np.eye(len(q)) - Jp @ J) @ (null_gain * (pref - q))
            q, _, _ = solve_ik(p, q, links, lam=lam, limit=limit, tol=tol, max_iter=50)
        out.append(q.copy())
    return np.array(out)


def path_error(points, qs, links=LINKS):
    """Largest distance between the wanted points and forward kinematics of the joint angles (metres)."""
    return float(max(np.linalg.norm(np.asarray(p) - fk(q, links)) for p, q in zip(points, qs)))

## Step 5: failure cases and sensitivity
(a) an unreachable target: the iteration settles on the closest point it can get to;
(b) joint limits can make a reachable point unreachable; (c) a link-length error moves the
gripper by about the same amount.

In [ ]:
def multi_start(target, links=LINKS, limit=None, starts=24, seed=0, lam=0.01, max_iter=300):
    """Best (smallest error) result over random starting poses."""
    rng = np.random.default_rng(seed)
    lim = np.pi if limit is None else limit
    best = None
    for _ in range(starts):
        q0 = rng.uniform(-lim, lim, len(links))
        q, err, _ = solve_ik(target, q0, links, lam=lam, limit=limit, max_iter=max_iter)
        if best is None or err < best[1]:
            best = (q, err)
    return best


def numeric_reach_map(limit, links=LINKS, cell=0.1, starts=5, tol=1e-6, max_iter=80):
    """Points of the disk whose multi-start numerical IK converges (error < tol); returns (xs, ys, mask)."""
    r_max = float(np.sum(links))
    xs = -r_max + (np.arange(int(np.ceil(2 * r_max / cell))) + 0.5) * cell
    mask = np.zeros((len(xs), len(xs)), dtype=bool)
    for i, y in enumerate(xs):
        for j, x in enumerate(xs):
            if np.hypot(x, y) <= r_max:
                mask[i, j] = multi_start((x, y), links, limit, starts, max_iter=max_iter)[1] < tol
    return xs, xs, mask


def link_error_effect(q, dl, links=LINKS):
    """Distance (m) the gripper moves when the link lengths are really links + dl instead of links."""
    return float(np.linalg.norm(fk(q, np.asarray(links) + np.asarray(dl)) - fk(q, links)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q = np.radians([30, 40, -20])
    print("max |J - J_fd| =", np.abs(jacobian(q) - jacobian_fd(q)).max())
    a, b = np.array([0.25, -0.30]), np.array([0.25, 0.30])
    pts = line_path(a, b, 41)
    q_start = solve_ik(pts[0], [0.3, -0.3, 0.3])[0]
    plain = track_path(pts, q_start)
    nul = track_path(pts, q_start, null_gain=0.1)
    print("path error (m): plain", path_error(pts, plain), " null-space", path_error(pts, nul))
    print("largest joint angle (degrees): plain", np.degrees(np.abs(plain).max()).round(1),
          " null-space", np.degrees(np.abs(nul).max()).round(1))
    for eps in (0.1, 0.01, 0.001):
        qq = np.array([0.0, eps, 0.0])
        print(f"bend {eps}: s_min {singular_values(qq)[-1]:.5f}, pinv speed {np.linalg.norm(joint_speeds(qq, [0.1, 0])):.2f}, "
              f"damped speed {np.linalg.norm(joint_speeds(qq, [0.1, 0], lam=0.05)):.2f} rad/s")

    t = np.linspace(0, 1, 41)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, qs, name in ((axs[0], plain, "plain tracking"), (axs[1], nul, "with null-space motion")):
        for k in range(3):
            ax.plot(t, np.degrees(qs[:, k]), label=f"q{k + 1}")
        ax.axhline(120, color="gray", ls=":")
        ax.axhline(-120, color="gray", ls=":")
        ax.set_xlabel("path time t / T")
        ax.set_title(f"Joint angles, {name}")
        ax.grid(alpha=0.3)
    axs[0].set_ylabel("joint angle (degrees); dotted: +-120 limits")
    axs[0].legend()
    fig.tight_layout()
    fig.savefig(OUT / "path_joints.png", dpi=110)

    eps = np.logspace(-4, -0.5, 40)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.loglog(eps, [np.linalg.norm(joint_speeds([0, e, 0], [0.1, 0])) for e in eps], label="pseudoinverse")
    ax2.loglog(eps, [np.linalg.norm(joint_speeds([0, e, 0], [0.1, 0], lam=0.05)) for e in eps], label="damped, lambda 0.05")
    ax2.set_xlabel("elbow bend away from straight (rad)")
    ax2.set_ylabel("joint speed norm (rad/s) for 0.1 m/s along the arm")
    ax2.set_title("Speeds blow up near the stretched singularity")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "singularity_speeds.png", dpi=120)
    q_far, err_far, _ = solve_ik((0.9, 0.0), [0.3, 0.3, 0.3])
    print("unreachable target (0.9, 0): gripper", fk(q_far).round(4), "error", round(err_far, 4))
    _, e120 = multi_start((-0.50, 0.0), limit=np.radians(120))
    _, e180 = multi_start((-0.50, 0.0))
    print("target (-0.50, 0.0): error with +-120 limit", round(e120, 4), " without limits", e180)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Print `solve_ik((0.30, 0.40), [0.3, 0.3, 0.3])` and check the result with `fk`. Try `lam=0.3` and compare the iteration count with `lam=0.01`.
2. **Stretch.** Use `joint_speeds` to find, for bends 0.5 and 0.05 rad, the pseudoinverse speed for 0.1 m/s along the arm, and compare with 0.1 / smallest singular value.
3. **Explain.** Explain why a path that starts at the pick point with q2 = -122 degrees may violate a 120 degree limit even though the gripper positions are all reachable.

## Assessment
- **A. Conceptual.** Why can a pure pseudoinverse demand enormous joint speeds near a fully stretched arm, and how does damping fix this? What does damping cost?
- **B. Numerical.** Write the Jacobian of the stretched arm (all angles 0) with L = (0.30, 0.25, 0.15) m, and find its rank, its singular values and the manipulability.
- **C. Graph and simulation interpretation.** In the log-log figure, what is the slope of the pseudoinverse line and what does it mean for the speed when the bend is ten times smaller? What does the damped line do?
- **D. Design and optimisation challenge.** Design: plain tracking of the path from (0.25, -0.30) m to (0.25, 0.30) m (a start with q2 near -122 degrees) swings a joint to 150.7 degrees, but the motors stop at 120 degrees. Give two ways to fix it, with numbers, and one trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def _link_distance(L, q, c):
    """Smallest distance from point c to any link segment of the arm at joint angles q."""
    cum = np.cumsum(q)
    P = np.vstack([[0, 0], np.cumsum(L.LINKS[:, None] * np.stack([np.cos(cum), np.sin(cum)], axis=1), axis=0)])
    best = 1e9
    for i in range(3):
        ab = P[i + 1] - P[i]
        t = np.clip((c - P[i]) @ ab / (ab @ ab), 0, 1)
        best = min(best, float(np.linalg.norm(c - (P[i] + t * ab))))
    return best


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    d = np.radians
    q = d([30, 40, -20])
    checks.append(("Jacobian at (30, 40, -20) degrees matches finite differences (error below 1e-7)",
                   np.abs(L.jacobian(q) - L.jacobian_fd(q)).max() < 1e-7))
    checks.append(("straight arm: Jacobian is [[0, 0, 0], [0.70, 0.40, 0.15]] by hand",
                   np.allclose(L.jacobian(d([0, 0, 0])), [[0, 0, 0], [0.70, 0.40, 0.15]])))
    checks.append(("straight arm: second singular value is 0 and manipulability is 0",
                   L.singular_values([0, 0, 0])[1] < 1e-12 and L.manipulability([0, 0, 0]) < 1e-12))
    q0 = np.array([0.3, 0.3, 0.3])
    qs, err, it = L.solve_ik((0.45, 0.20), q0)
    checks.append(("worked example: IK reaches (0.45, 0.20) m within 1e-9 m in 11 iterations",
                   err < 1e-9 and it == 11 and np.allclose(L.fk(qs), [0.45, 0.20], atol=1e-9)))
    e0 = np.array([0.45, 0.20]) - L.fk(q0)
    raw = L.dls_step(L.jacobian(q0), e0, 0.01)
    errs = [L.solve_ik((0.45, 0.20), q0, max_iter=k)[1] for k in (0, 1, 8, 9, 10)]
    checks.append(("worked example: start error 0.2006 m, first raw step 1.56 rad capped to 0.1 rad (error 0.1861 m)",
                   abs(np.linalg.norm(e0) - 0.2006) < 1e-4 and abs(np.abs(raw).max() - 1.56) < 0.01
                   and abs(errs[0] - 0.2006) < 1e-4 and abs(errs[1] - 0.1861) < 1e-4))
    checks.append(("worked example: errors after 8, 9, 10 iterations are 4.3e-4, 9.5e-7, 1.9e-9 m (fast final convergence)",
                   abs(errs[2] - 4.33e-4) < 1e-5 and abs(errs[3] - 9.5e-7) < 1e-7 and abs(errs[4] - 1.9e-9) < 2e-10))
    e25 = [L.solve_ik((0.45, 0.20), q0, max_iter=k)[1] for k in (2, 5)]
    checks.append(("worked example: start at (0.586, 0.347) m, raw step (-1.28, 1.56, 1.50) rad, errors 0.168 m after 2 and 0.091 m after 5 iterations",
                   np.allclose(L.fk(q0), [0.586, 0.347], atol=1e-3) and np.allclose(raw, [-1.283, 1.560, 1.495], atol=0.005)
                   and abs(e25[0] - 0.168) < 1e-3 and abs(e25[1] - 0.091) < 1e-3))
    checks.append(("worked example: final angles (-21.0, 60.7, 59.0) degrees",
                   np.allclose(np.degrees(L.solve_ik((0.45, 0.20), q0)[0]), [-21.05, 60.67, 59.02], atol=0.05)))
    checks.append(("stretched arm: largest singular value sqrt(0.70^2 + 0.40^2 + 0.15^2) = 0.82 m/rad",
                   abs(L.singular_values([0, 0, 0])[0] - np.sqrt(0.49 + 0.16 + 0.0225)) < 1e-12 and abs(L.singular_values([0, 0, 0])[0] - 0.82) < 1e-3))
    Jq = L.jacobian(q)
    checks.append(("a DLS step equals J^T (J J^T + lambda^2 I)^-1 e written out by hand",
                   np.allclose(L.dls_step(Jq, np.array([0.01, -0.02]), 0.05),
                               Jq.T @ np.linalg.inv(Jq @ Jq.T + 0.0025 * np.eye(2)) @ [0.01, -0.02])))
    speeds = []
    for eps in (0.1, 0.01, 0.001):
        qq = [0.0, eps, 0.0]
        speeds.append((L.singular_values(qq)[1], np.linalg.norm(L.joint_speeds(qq, [0.1, 0.0])),
                       np.linalg.norm(L.joint_speeds(qq, [0.1, 0.0], lam=0.05))))
    checks.append(("smallest singular value is 0.0156, 0.00156, 0.00016 for bends of 0.1, 0.01, 0.001 rad",
                   np.allclose([s[0] for s in speeds], [0.01562, 0.001562, 0.0001562], rtol=0.01)))
    checks.append(("pseudoinverse speeds for 0.1 m/s along the arm: 6.4, 64, 640 rad/s; damped (0.05): 0.57, 0.06, 0.01 rad/s",
                   np.allclose([s[1] for s in speeds], [6.39, 63.99, 639.87], rtol=0.01)
                   and np.allclose([s[2] for s in speeds], [0.57, 0.06, 0.01], atol=0.006)))
    checks.append(("pseudoinverse speed is about 0.1 / smallest singular value",
                   all(abs(s[1] * s[0] - 0.1) < 0.002 for s in speeds)))
    checks.append(("manipulability equals sqrt(det(J J^T))",
                   abs(L.manipulability(q) - np.sqrt(np.linalg.det(Jq @ Jq.T))) < 1e-12))
    grid = d(np.arange(0, 181, 15))
    best = max(L.manipulability([a, b, c]) for a in grid for b in grid for c in grid)
    checks.append(("largest manipulability on a 15 degree grid is about 0.133 m^2, the straight arm has 0", abs(best - 0.133) < 0.003))
    checks.append(("quintic timing: 0 at start, 1 at end, 0.5 in the middle, flat at both ends",
                   abs(L.quintic(0.0)) < 1e-15 and abs(L.quintic(1.0) - 1) < 1e-12 and abs(L.quintic(0.5) - 0.5) < 1e-12
                   and abs(L.quintic(1e-3)) < 1e-7 and abs(1 - L.quintic(1 - 1e-3)) < 1e-7))
    n = 401
    a_, b_ = np.array([0.25, -0.30]), np.array([0.25, 0.30])
    peak_q = np.linalg.norm(np.diff(L.line_path(a_, b_, n), axis=0), axis=1).max() / (2.0 / (n - 1))
    peak_c = np.linalg.norm(np.diff(L.line_path(a_, b_, n, L.cubic), axis=0), axis=1).max() / (2.0 / (n - 1))
    checks.append(("2 s move of 0.60 m: peak gripper speed 0.5625 m/s (quintic, 1.875 x mean) and 0.45 m/s (cubic, 1.5 x mean)",
                   abs(peak_q - 0.5625) < 1e-3 and abs(peak_c - 0.45) < 1e-3))
    pts = L.line_path(a_, b_, 41)
    qa = L.solve_ik(pts[0], [0.3, -0.3, 0.3])[0]
    plain = L.track_path(pts, qa)
    nul = L.track_path(pts, qa, null_gain=0.1)
    checks.append(("path validated: gripper error below 1e-6 m with and without null-space motion",
                   L.path_error(pts, plain) < 1e-6 and L.path_error(pts, nul) < 1e-6))
    checks.append(("joint limits: plain tracking swings a joint to 150.7 degrees, null-space motion keeps it to 102.8 degrees",
                   abs(np.degrees(np.abs(plain).max()) - 150.7) < 0.1 and abs(np.degrees(np.abs(nul).max()) - 102.8) < 0.1))
    checks.append(("the 2 s plain path peaks at 2.42 rad/s in one joint",
                   abs(np.abs(np.diff(plain, axis=0)).max() / (2.0 / 40) - 2.42) < 0.02))
    other = L.track_path(pts, L.solve_ik(pts[0], [0.3, 0.3, 0.3])[0])
    checks.append(("another elbow choice at the start (from q = 0.3, 0.3, 0.3) keeps plain tracking below 99.5 degrees",
                   abs(np.degrees(np.abs(other).max()) - 99.5) < 0.1 and L.path_error(pts, other) < 1e-6))
    warm = np.degrees(np.abs(np.diff(plain, axis=0)).max())
    alt = np.array([L.solve_ik(p, [0.3, -0.3, 0.3] if i % 2 else [0.3, 0.3, 0.3])[0] for i, p in enumerate(pts)])
    checks.append(("elbow flip: warm starts change a joint at most 6.9 degrees per step, alternating cold starts jump 249 degrees",
                   abs(warm - 6.93) < 0.1 and abs(np.degrees(np.abs(np.diff(alt, axis=0)).max()) - 249.0) < 1.0
                   and L.path_error(pts, alt) < 1e-6))
    qf, errf, _ = L.solve_ik((0.9, 0.0), q0)
    checks.append(("unreachable target (0.9, 0): the gripper stops within 2 mm of the boundary point (0.70, 0), error 0.2007 m",
                   abs(np.linalg.norm(L.fk(qf)) - 0.70) < 0.002 and abs(errf - 0.2007) < 0.001))
    q_lim, e_lim = L.multi_start((-0.50, 0.0), limit=d(120))
    q_free, e_free = L.multi_start((-0.50, 0.0))
    checks.append(("target (-0.50, 0) is reachable without limits but misses by 5.6 cm with +-120 degree limits",
                   e_free < 1e-9 and abs(e_lim - 0.0557) < 0.002 and np.abs(q_lim).max() <= d(120) + 1e-12))
    xs, _, mask = L.numeric_reach_map(d(120))
    inside = np.hypot(*np.meshgrid(xs, xs)) <= 0.70
    checks.append(("numerical reach map at +-120 degrees: 156 cells inside the disk, 0.859 of them reached",
                   inside.sum() == 156 and abs(mask[inside].mean() - 0.859) < 0.002))
    qq = L.solve_ik((0.45, 0.20), q0)[0]
    e1 = [L.link_error_effect(qq, np.eye(3)[i] * 0.001) * 1000 for i in range(3)]
    checks.append(("1 mm error in any single link moves the gripper by 1.000 mm", np.allclose(e1, 1.0, atol=1e-6)))
    checks.append(("1 mm error in all three links: 3.000 mm when stretched, 2.005 mm at the pose for (0.45, 0.20)",
                   abs(L.link_error_effect(np.zeros(3), [0.001] * 3) * 1000 - 3.0) < 1e-6
                   and abs(L.link_error_effect(qq, [0.001] * 3) * 1000 - 2.005) < 0.001))
    A, B, C, R = np.array([0.40, -0.20]), np.array([0.40, 0.20]), np.array([0.40, 0.0]), 0.06
    pv = C + np.array([0.07, 0.0])
    route = np.vstack([L.line_path(A, pv, 41)[:-1], L.line_path(pv, B, 41)])
    qr = L.track_path(route, L.solve_ik(A, q0, limit=d(120))[0], limit=d(120), null_gain=0.1)
    dense = np.vstack([A + (pv - A) * t for t in np.linspace(0, 1, 200)] + [pv + (B - pv) * t for t in np.linspace(0, 1, 200)])
    checks.append(("no-go zone (centre (0.40, 0), radius 0.06): the straight route hits the centre, a 0.07 m detour clears it and tracks within 1e-6 m",
                   min(np.linalg.norm(p - C) for p in L.line_path(A, B, 401)) < 1e-9
                   and min(np.linalg.norm(p - C) for p in dense) > R and L.path_error(route, qr) < 1e-6
                   and np.abs(qr).max() <= d(120) + 1e-9))
    checks.append(("a clearance margin of 0.03 m is not kept for the links on this route",
                   min(_link_distance(L, qv, C) for qv in qr) < R + 0.03))
    checks.append(("the route checks the gripper only: at some pose a link passes within 0.06 m of the zone centre",
                   min(_link_distance(L, qv, C) for qv in qr) < R))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
